This PySpark code dynamically generates a table name by formatting yesterday's date in the `MMddyyyy` format and appending it to the prefix `"Snapshot_PM_"`. It also sets legacy datetime rebasing configurations for reading and writing Parquet files. Finally, it prints the generated table name.

In [ ]:
%%pyspark
from pyspark.sql.functions import date_format, current_date, date_sub

# Configure legacy datetime rebasing for Parquet files (for compatibility with older data formats)
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")


# Get yesterday's date formatted as MMddyyyy
formatted_date = spark.sql(
    "SELECT date_format(current_date() - INTERVAL 1 DAY, 'MMddyyyy') AS formatted_date"
).collect()[0]['formatted_date']

# Dynamically construct the table name using the formatted date
table_name = f"Snapshot_PM_{formatted_date}"

# Output the generated table name
print(f"Generated Table Name: {table_name}")


StatementMeta(, , -1, SessionError, , SessionError)

InvalidHttpRequestToLivy: [TooManyRequestsForCapacity] This spark job can't be run because you have hit a spark compute or API rate limit. To run this spark job, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. HTTP status code: 430 {Learn more} HTTP status code: 430.

**Josh code pt1:** Run SSRS "PM Snapshot - MonthEnd" report

In [ ]:
%%pyspark

# Define the query
query = """
SELECT
    CASE 
        WHEN Responsibility_Center LIKE '%TRANSIT%' THEN 40
        ELSE Responsibility_Center
    END AS CSC,
    Description,
    `Next_PM_Date`,
    `Maintenance_Contract`,
    `Contract_Type`,
    `Next_PM_Action`,
    `Last_PM_Date`,
    No_,
    `Customer_No_`,
    `Customer_Name`,
    `Fixed_Schedule_Interval_Days`,
    Blocked
FROM
    `Equipment_Object` AS Equipment_Object
WHERE
    `Next_PM_Date` > '1900-01-01'
    AND Blocked = 0
    AND CAST(`Next_PM_Date` AS DATE) < CAST(DATE_ADD(current_date(), 30) AS DATE)
ORDER BY `Next_PM_Date` ASC
"""

# Execute the query
df = spark.sql(query)

# Display row count
row_count = df.count()
print(f"Row count: {row_count}")

# Show the first few rows of the result, if needed
display(df)

df.write.format("delta").mode("overwrite").saveAsTable("PMDue")

StatementMeta(, , -1, Cancelled, , Cancelled)

**Josh code pt2:** Run "PM Completion with SA"

In [ ]:
%%pyspark
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DateType
)

# Define the adjusted query without the `dbo_Production_` prefix
query = f"""
SELECT 
    PMDue.CSC AS CSC_Id,
    PMDue.Customer_No_ AS Sell_to_Customer_Number,
    PMDue.No_ AS Equipment_Object_Id,
    PMDue.Contract_Type,
    CASE 
        WHEN EObj.Last_PM_Date < '1900-01-01' THEN NULL
        ELSE EObj.Next_PM_Date
    END AS Next_PM_Date,
    CASE 
        WHEN EObj.Last_PM_Date < '1900-01-01' THEN NULL
        ELSE EObj.Last_PM_Date
    END AS Last_PM_Date,
    CASE 
        WHEN SAP.Date_Fixed = 1 THEN 'Bumped' 
        ELSE ''
    END AS Bumped,
    CASE 
        WHEN EObj.Next_PM_Date <= date_sub(current_date(), 1) THEN 'Still Due' 
        ELSE 'Done' 
    END AS Still_Due,
    PMDue.Fixed_Schedule_Interval_Days,
    EObj.Default_Rental_Return_Location,
    LAST_DAY(CURRENT_DATE - INTERVAL 1 MONTH) AS End_of_Month,
    '{table_name}' AS File_Name
FROM 
    PMDue 
    INNER JOIN Equipment_Object EObj ON PMDue.No_ = EObj.No_
    INNER JOIN Service_Action_Planning SAP ON EObj.No_ = SAP.Equipment_Object AND EObj.Next_PM_Date = SAP.Date
    INNER JOIN Responsibility_Center_ELC RCE ON PMDue.CSC = RCE.Code
    LEFT JOIN PMType ON PMDue.Contract_Type = PMType.Contract_Type
"""

# Define the schema
schema = StructType([
    StructField("CSC_Id", LongType(), True),  # BIGINT
    StructField("Sell_to_Customer_Number", StringType(), True),  # STRING
    StructField("Equipment_Object_Id", StringType(), True),  # STRING
    StructField("Contract_Type", StringType(), True),  # STRING
    StructField("Next_PM_Date", DateType(), True),  # DATE
    StructField("Last_PM_Date", DateType(), True),  # DATE
    StructField("Bumped", StringType(), True),  # STRING
    StructField("Still_Due", StringType(), True),  # STRING
    StructField("Fixed_Schedule_Interval_Days", LongType(), True),  # BIGINT
    StructField("Default_Rental_Return_Location", StringType(), True),  # STRING
    StructField("End_of_Month", DateType(), True),  # DATE
    StructField("File_Name", StringType(), True),  # STRING
])

# Execute the query and apply the schema
df_josh = spark.sql(query)

# Cast the DataFrame to the defined schema
df_josh = df_josh.select(
    df_josh["CSC_Id"].cast("bigint").alias("CSC_Id"),
    df_josh["Sell_to_Customer_Number"].cast("string").alias("Sell_to_Customer_Number"),
    df_josh["Equipment_Object_Id"].cast("string").alias("Equipment_Object_Id"),
    df_josh["Contract_Type"].cast("string").alias("Contract_Type"),
    df_josh["Next_PM_Date"].cast("date").alias("Next_PM_Date"),
    df_josh["Last_PM_Date"].cast("date").alias("Last_PM_Date"),
    df_josh["Bumped"].cast("string").alias("Bumped"),
    df_josh["Still_Due"].cast("string").alias("Still_Due"),
    df_josh["Fixed_Schedule_Interval_Days"].cast("bigint").alias("Fixed_Schedule_Interval_Days"),
    df_josh["Default_Rental_Return_Location"].cast("string").alias("Default_Rental_Return_Location"),
    df_josh["End_of_Month"].cast("date").alias("End_of_Month"),
    df_josh["File_Name"].cast("string").alias("File_Name"),
)


# Display row count
row_count = df_josh.count()
print(f"Row count: {row_count}")

# Show the first few rows
df_josh.show()


StatementMeta(, , -1, Cancelled, , Cancelled)

In [ ]:
%%pyspark

df_josh.write.format("delta").mode("overwrite").saveAsTable(table_name)
df_josh.write.format("delta").mode("append").saveAsTable("PM_Completion_Snapshots")

# If snapshot missed, use time travel

In [ ]:
%%pyspark
# Query Delta table history using SQL
table_name = "Equipment_Object"  # replace with your actual table name
history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")
history_df.show(truncate=False)

StatementMeta(, ec50a4b4-9b32-4920-8a7a-85bb4582703a, 3, Finished, Available, Finished, False)

+-------+-----------------------+------+--------+------------+-------------------+----+--------+---------+-----------+--------------+-------------+----------------+------------+-------------------------+
|version|timestamp              |userId|userName|operation   |operationParameters|job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics|userMetadata|engineInfo               |
+-------+-----------------------+------+--------+------------+-------------------+----+--------+---------+-----------+--------------+-------------+----------------+------------+-------------------------+
|1322   |2026-04-06 09:06:51.504|NULL  |NULL    |Update      |NULL               |NULL|NULL    |NULL     |NULL       |NULL          |NULL         |NULL            |NULL        |DataFactoryCopy_3.0.2.300|
|1321   |2026-04-06 09:05:56.615|NULL  |NULL    |ReplaceTable|NULL               |NULL|NULL    |NULL     |NULL       |NULL          |NULL         |NULL            |NULL        |DataFac

In [7]:
%%pyspark
from pyspark.sql.functions import (
    date_format, current_date, date_sub, date_add, to_date,
    when, col, lit, last_day, add_months, desc
)

# Configure legacy datetime rebasing for Parquet files (for compatibility with older data formats)
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")

# Option to set a custom base date for time travel.
# Set custom_date_str to a valid date string (e.g., "2025-04-07") to time travel to that day,
# or set to None to use the current date and current data.
custom_date_str = "2025-04-01"  # <-- Change this value as needed, or set to None

# Determine the base date: if a custom date is provided, use it; otherwise, use current_date()
if custom_date_str:
    base_date = to_date(lit(custom_date_str), "yyyy-MM-dd")
else:
    base_date = current_date()

# Get yesterday's date based on the base_date and format it as MMddyyyy for dynamic table naming
yesterday = date_sub(base_date, 1)
formatted_date = spark.range(1) \
    .select(date_format(yesterday, "MMddyyyy").alias("formatted_date")) \
    .collect()[0]["formatted_date"]

# Construct the dynamic table name
dynamic_table_name = f"Snapshot_PM_{formatted_date}"
print(f"Generated Table Name: {dynamic_table_name}")

# Define a function to read a Delta table using time travel based on a custom date.
# This function queries the table's history, filters for commits on the specified date,
# orders them to get the latest commit, and then uses that commit timestamp for time travel.
def read_delta_table_with_time_travel(table_name: str, custom_date_str: str = None):
    if custom_date_str:
        # Query the Delta table history using DESCRIBE HISTORY
        history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")
        # Filter the history for records on the provided custom date
        filtered_history = history_df.filter(to_date(col("timestamp")) == custom_date_str)
        # Order the filtered history by timestamp descending to get the latest commit for that day
        latest_commit = filtered_history.orderBy(desc("timestamp")).limit(1).collect()
        if latest_commit:
            commit_timestamp = latest_commit[0]["timestamp"]
            print(f"Using commit timestamp: {commit_timestamp} for table {table_name}")
            return spark.read.format("delta").option("timestampAsOf", commit_timestamp).table(table_name)
        else:
            raise ValueError(f"No commit found for date: {custom_date_str} in table {table_name}")
    else:
        return spark.table(table_name)

# ---------------------------------------
# Section 1: Create PMDue table using Equipment_Object table
# ---------------------------------------

# Read the Equipment_Object table using time travel
df_equip = read_delta_table_with_time_travel("Equipment_Object", custom_date_str)

# Create the PMDue DataFrame:
#  - Replace Responsibility_Center values containing 'TRANSIT' with 40
#  - Filter rows: Next_PM_Date > '1900-01-01', Blocked = 0,
#    and Next_PM_Date is less than base_date + 30 days
df_pmdue = (
    df_equip.withColumn(
        "CSC",
        when(col("Responsibility_Center").like("%TRANSIT%"), lit(40))
        .otherwise(col("Responsibility_Center"))
    )
    .filter(
        (col("Next_PM_Date") > lit("1900-01-01")) &
        (col("Blocked") == 0) &
        (to_date(col("Next_PM_Date")) < to_date(date_add(base_date, 30)))
    )
    .select(
        "CSC", "Description", "Next_PM_Date", "Maintenance_Contract", 
        "Contract_Type", "Next_PM_Action", "Last_PM_Date", "No_", 
        "Customer_No_", "Customer_Name", "Fixed_Schedule_Interval_Days", "Blocked"
    )
    .orderBy("Next_PM_Date")
)

# Display row count and preview the PMDue DataFrame
row_count = df_pmdue.count()
print(f"Row count: {row_count}")
display(df_pmdue)

# Write PMDue as a Delta table
df_pmdue.write.format("delta").mode("overwrite").saveAsTable("PMDue")


# ---------------------------------------
# Section 2: Join multiple tables using time travel
# ---------------------------------------

# Read required tables using the same time travel function
df_pmdue  = read_delta_table_with_time_travel("PMDue")
df_eobj   = read_delta_table_with_time_travel("Equipment_Object", custom_date_str)
df_sap    = read_delta_table_with_time_travel("Service_Action_Planning", custom_date_str)
df_rce    = read_delta_table_with_time_travel("Responsibility_Center_ELC")
df_pmtype = read_delta_table_with_time_travel("PMType")

# Build the joined DataFrame
df_josh = (
    df_pmdue.alias("pmdue")
    .join(df_eobj.alias("eobj"), col("pmdue.No_") == col("eobj.No_"), "inner")
    .join(
        df_sap.alias("sap"),
        (col("eobj.No_") == col("sap.Equipment_Object")) &
        (col("eobj.Next_PM_Date") == col("sap.Date")),
        "inner"
    )
    .join(df_rce.alias("rce"), col("pmdue.CSC") == col("rce.Code"), "inner")
    .join(df_pmtype.alias("pmtype"), col("pmdue.Contract_Type") == col("pmtype.Contract_Type"), "left")
    .select(
        col("pmdue.CSC").alias("CSC_Id"),
        col("pmdue.Customer_No_").alias("Sell_to_Customer_Number"),
        col("pmdue.No_").alias("Equipment_Object_Id"),
        col("pmdue.Contract_Type").alias("Contract_Type"),
        # Set Next_PM_Date to null if Last_PM_Date is before 1900-01-01; otherwise, use eobj's value
        when(col("eobj.Last_PM_Date") < lit("1900-01-01"), None)
            .otherwise(col("eobj.Next_PM_Date")).alias("Next_PM_Date"),
        # Similarly for Last_PM_Date
        when(col("eobj.Last_PM_Date") < lit("1900-01-01"), None)
            .otherwise(col("eobj.Last_PM_Date")).alias("Last_PM_Date"),
        # Determine Bumped status based on SAP.Date_Fixed flag
        when(col("sap.Date_Fixed") == 1, lit("Bumped"))
            .otherwise(lit("")).alias("Bumped"),
        # Mark as 'Still Due' if eobj.Next_PM_Date is on or before (base_date - 1); otherwise, 'Done'
        when(col("eobj.Next_PM_Date") <= date_sub(base_date, 1), lit("Still Due"))
            .otherwise(lit("Done")).alias("Still_Due"),
        col("pmdue.Fixed_Schedule_Interval_Days").alias("Fixed_Schedule_Interval_Days"),
        col("eobj.Default_Rental_Return_Location").alias("Default_Rental_Return_Location"),
        # Calculate End_of_Month as the last day of the month before base_date
        last_day(add_months(base_date, -1)).alias("End_of_Month"),
        lit(dynamic_table_name).alias("File_Name")
    )
)

# Cast columns to match the required schema types
df_josh = df_josh.select(
    col("CSC_Id").cast("bigint"),
    col("Sell_to_Customer_Number").cast("string"),
    col("Equipment_Object_Id").cast("string"),
    col("Contract_Type").cast("string"),
    col("Next_PM_Date").cast("date"),
    col("Last_PM_Date").cast("date"),
    col("Bumped").cast("string"),
    col("Still_Due").cast("string"),
    col("Fixed_Schedule_Interval_Days").cast("bigint"),
    col("Default_Rental_Return_Location").cast("string"),
    col("End_of_Month").cast("date"),
    col("File_Name").cast("string")
)

# Display row count and preview the joined DataFrame
row_count = df_josh.count()
print(f"Row count: {row_count}")
df_josh.show()

# Write the joined DataFrame to the dynamically generated table and append to PM_Completion_Snapshots
df_josh.write.format("delta").mode("overwrite").saveAsTable(dynamic_table_name)
df_josh.write.format("delta").mode("append").saveAsTable("PM_Completion_Snapshots")


StatementMeta(, f0e07d56-ba3c-4267-8185-a426f937145e, 9, Finished, Available, Finished)

Generated Table Name: Snapshot_PM_03312025
Using commit timestamp: 2025-04-01 09:04:41.798000 for table Equipment_Object
Row count: 12326


SynapseWidget(Synapse.DataFrame, 1533c2b9-5028-4bc2-afa6-7be4c12446b6)

Using commit timestamp: 2025-04-01 09:04:41.798000 for table Equipment_Object
Using commit timestamp: 2025-04-01 09:09:43.667000 for table Service_Action_Planning
Row count: 12328
+------+-----------------------+-------------------+-------------+------------+------------+------+---------+----------------------------+------------------------------+------------+--------------------+
|CSC_Id|Sell_to_Customer_Number|Equipment_Object_Id|Contract_Type|Next_PM_Date|Last_PM_Date|Bumped|Still_Due|Fixed_Schedule_Interval_Days|Default_Rental_Return_Location|End_of_Month|           File_Name|
+------+-----------------------+-------------------+-------------+------------+------------+------+---------+----------------------------+------------------------------+------------+--------------------+
|     7|              183070001|            E172544|         T360|  2025-02-17|  2024-11-19|      |Still Due|                          90|                             7|  2025-03-31|Snapshot_PM_03312025|
|   

# Other

In [7]:
%%pyspark

# Count total rows in the table
row_count = df_josh.count()

# Count rows based on conditions
done_count = df_josh.filter(df_josh["Still_Due"] == "Done").count()       # Count rows marked as "Done"
still_due_count = df_josh.filter(df_josh["Still_Due"] == "Still Due").count() # Count rows marked as "Still Due"
bumped_count = df_josh.filter(df_josh["Bumped"] == 'Bumped').count()          # Count rows with "Bumped" set to 1

# Output row counts
print(f"Row count: {row_count}")
print(f"Done count: {done_count}")
print(f"Still Due count: {still_due_count}")
print(f"Bumped count: {bumped_count}")

# Calculate the percentage of tasks marked as "Done"
total_count = still_due_count + done_count
percentage_done = (done_count / total_count) * 100 if total_count > 0 else 0.0  # Avoid division by zero

# Output the percentage of completed tasks
print(f"% Done: {percentage_done:.2f}%")

StatementMeta(, 702ee20f-5499-429f-856c-b37f6c2f3546, 9, Finished, Available, Finished)

Row count: 14602
Done count: 12721
Still Due count: 1881
Bumped count: 2827
% Done: 87.12%


In [8]:
%%pyspark

# Execute a DELETE SQL command on the Delta table
spark.sql("""
DELETE FROM PM_Completion_Snapshots
WHERE End_of_Month = '2024-11-30'
""")

StatementMeta(, 702ee20f-5499-429f-856c-b37f6c2f3546, 10, Finished, Available, Finished)

DataFrame[num_affected_rows: bigint]